In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
## TFlite Introduction
## TFlite acheives two things: 
# 1. Make models smaller so they can fit on devices with limited storage
# 2. Make models faster so they can run efficiently on devices with limited compute power

# TFlite can be divided into two main sections:
# 1. TFlite Converter - Converts standard TensorFlow models into the TFlite format
# 2. TFlite Interpreter - Runs TFlite models on varous runtime environments: ANdroid, iOS, embedded Linux, microcontrollers etc.

In [3]:
# Train model

l0 = Dense(units=1, input_shape=[1])
model = Sequential([l0])
model.compile(optimizer='sgd', loss='mean_squared_error')
xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-3.0, -1.0, 1.0, 3.0, 5.0, 7.0], dtype=float)
model.fit(xs, ys, epochs=500)

Epoch 1/500
1/1 [==============================] - 1s 905ms/step - loss: 23.0800
Epoch 2/500
1/1 [==============================] - 0s 11ms/step - loss: 18.4396
Epoch 3/500
1/1 [==============================] - 0s 13ms/step - loss: 14.7829
Epoch 4/500
1/1 [==============================] - 0s 0s/step - loss: 11.9003
Epoch 5/500
1/1 [==============================] - 0s 6ms/step - loss: 9.6269
Epoch 6/500
1/1 [==============================] - 0s 2ms/step - loss: 7.8329
Epoch 7/500
1/1 [==============================] - 0s 6ms/step - loss: 6.4161
Epoch 8/500
1/1 [==============================] - 0s 7ms/step - loss: 5.2963
Epoch 9/500
1/1 [==============================] - 0s 0s/step - loss: 4.4101
Epoch 10/500
1/1 [==============================] - 0s 13ms/step - loss: 3.7079
Epoch 11/500
1/1 [==============================] - 0s 7ms/step - loss: 3.1506
Epoch 12/500
1/1 [==============================] - 0s 5ms/step - loss: 2.7073
Epoch 13/500
1/1 [==============================] - 0s

In [4]:
print(model.predict(np.array([[10.0]])))
print("Here is what I learned: {}".format(l0.get_weights()))

1/1 [==============================] - 0s 119ms/step
[[18.980127]]
Here is what I learned: [array([[1.9971197]], dtype=float32), array([-0.99107], dtype=float32)]


In [5]:
# Convert the model directly to TFlite format without quantization

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []  # Disable quantization to preserve precision

print("Converting model to TFLite format...")
tflite_model = converter.convert()
print(f"Conversion complete. Model size: {len(tflite_model)} bytes")

# Save to file
import pathlib
tflite_model_file = pathlib.Path('model.tflite')
tflite_model_file.write_bytes(tflite_model)
print("Model saved to model.tflite")

Converting model to TFLite format...
INFO:tensorflow:Assets written to: C:\Users\YUSUFS~1\AppData\Local\Temp\tmpidcipi3e\assets
Conversion complete. Model size: 1080 bytes
Model saved to model.tflite


In [6]:
## Load the model using tf.lite.Interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

# Get input and output tensor details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input Details:")
for detail in input_details:
    print(f"  Index: {detail['index']}, Shape: {detail['shape']}, Type: {detail['dtype']}")
    
print("\nOutput Details:")
for detail in output_details:
    print(f"  Index: {detail['index']}, Shape: {detail['shape']}, Type: {detail['dtype']}")

Input Details:
  Index: 0, Shape: [1 1], Type: <class 'numpy.float32'>

Output Details:
  Index: 3, Shape: [1 1], Type: <class 'numpy.float32'>


In [7]:
# Set input and invoke the interpreter
input_data = np.array([[10.0]], dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])


print(f"Input: {input_data[0][0]}")
print(f"TFLite Output: {output_data[0][0]}")

Input: 10.0
TFLite Output: 18.980127334594727


In [8]:
# Compare original model vs TFLite model
print("Original Keras Model Prediction:")
print(model.predict(np.array([[10.0]])))
print("\nTFLite Model Prediction:")
print(f"Output: {output_data}")

Original Keras Model Prediction:
1/1 [==============================] - 0s 33ms/step
[[18.980127]]

TFLite Model Prediction:
Output: [[18.980127]]


In [1]:
import numpy as np
import matplotlib.pylab as plt
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

import os
# define training and validation data directories
data_dir = 'horse-or-human/'
train_dir = os.path.join(data_dir, 'train/')
validation_dir = os.path.join(data_dir, 'validation/')

# classes are folders in each directory with these names
classes = ['horse','human']

In [2]:
## Keras ImageDataGenerator, data augmentation and validation set example

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale = 1/255,
                                   rotation_range = 40,
                                   width_shift_range = 0.2,
                                   height_shift_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True,
                                   fill_mode = 'nearest') # fill any missing pixels(due to zoom or rotation) with the values of their nearest pixels

validation_datagen = ImageDataGenerator(rescale = 1/255,
                                   rotation_range = 40,
                                   width_shift_range = 0.2,
                                   height_shift_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True,
                                   fill_mode = 'nearest')

train_generator = train_datagen.flow_from_directory(train_dir, target_size = (300,300), class_mode = 'binary')
validation_generator = train_datagen.flow_from_directory(validation_dir, target_size = (300,300), class_mode = 'binary')



Found 1027 images belonging to 2 classes.
Found 256 images belonging to 2 classes.


In [3]:
# create the feature extractor using the mobilenet_v2 module
module_selection = ("mobilenet_v2", 224, 1280) 
handle_base, pixels, FV_SIZE = module_selection
MODULE_HANDLE ="https://tfhub.dev/google/tf2-preview/{}/feature_vector/4".format(handle_base)
IMAGE_SIZE = (pixels, pixels)
feature_extractor = hub.KerasLayer(MODULE_HANDLE,
    input_shape=IMAGE_SIZE + (3,), 
    output_shape=[FV_SIZE],
    trainable=False)

In [ ]:
# Add a dense layer with 2 neurons (because we have 2 classes) for classification
model = tf.keras.Sequential([
    feature_extractor,
    tf.keras.layers.Dense(2, activation='softmax')
    ])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

history = model.fit(train_generator,
                    epochs=10,
                    validation_data=validation_generator)

In [ ]:
# Plot training & validation loss values
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')

plt.title('Model Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Convert the model directly to TFlite format without quantization

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []  # Disable quantization to preserve precision

print("Converting model to TFLite format...")
tflite_model = converter.convert()
print(f"Conversion complete. Model size: {len(tflite_model)} bytes")

# Save to file
import pathlib
tflite_model_file = pathlib.Path('model_cats_dogs.tflite')
tflite_model_file.write_bytes(tflite_model)
print("Model saved to model_cats_dogs.tflite")

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_model_file)
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]["index"]
output_index = interpreter.get_output_details()[0]["index"]

predictions = []

In [ ]:
## Test the TFlite model on 100 test images
test_labels, test_imgs = [], []

for img, label in test_batches.take(100):
    interpreter.set_tensor(input_index, img)
    interpreter.invoke()
    predictions.append(interpreter.get_tensor(output_index))
    test_labels.append(label.numpy()[0])
    test_imgs.append(img)
    
score = 0
for item in range(0,99):
    prediction=np.argmax(predictions[item])
    label = test_labels[item]
    if prediction==label:
        score=score+1
    
print("Out of 100 predictions I got " + str(score) + " correct")

In [ ]:
def plot_image(i, predictions_array, true_label, img):
    predictions = predictions_array[i]
    true_label = true_label[i]
    plt.imshow(img)
    predicted_label = np.argmax(predictions)
    if predicted_label == true_label:
        color = 'blue'
    else:
        color = 'red'
    plt.xlabel(f"{predicted_label} ({true_label})", color=color)

for index in range(0,99):
    plt.figure(figsize=(6,3))
    plt.subplot(1,2,1)
    plot_image(index, predictions, test_labels, test_imgs)
    plt.show()